## CONSTRUCTED 原始信号层方法

所有新计算值均为 **CONSTRUCTED 方法演示**，不是市场观测或历史验证。历史表格另行标记为重放。French 原始、提取和处理后的数据文件均未随包提供。French 路径：French 数据文件未随包提供，相关使用权和允许用途尚未明确，因此本 Notebook 不运行 French 数据分析。

以下代码与标签说明本 Notebook 中的构造信号分析。本 Notebook 不运行 French 市场数据分析。

[Project 4 读者指南](../../../../NOTEBOOK_GUIDE.md)


[本语言逐本运行模式与准备条件](../../../../NOTEBOOK_GUIDE.md)。运行步骤、数据要求和预期输出见本指南。

# Project 4 准备层：信号层（Value / Momentum / Quality）与构造公司宇宙

本 Notebook 调用 projects/project04/preparation/signal_layer.py 中的函数，在**构造**的公司宇宙上按点时（point-in-time, PIT）规则，选取每个月末当时已经可得的财报版本；计算字段契约规定的 7 个原始字段；依照冻结规则进行同日缩尾、行业与规模中性化、z 分数标准化和固定合成；并运行六个反例（将 period_end 当作 available_at、同一版本的重述处理规则、未来数据扰动、幸存者宇宙、全样本标准化、非正分母）。本 Notebook 使用当前 C13 信号算法。

- 构造设置：seed=20261210；240 家公司（其中 40 家银行）；96 个月。
- 所有数据均为教学用构造数据（生成过程已知，且未内置价值、动量或质量效应）；仓库中没有逐股点时财报、价格或历史股票池，因此结果不代表任何真实公司的观测。
- 本 Notebook 只覆盖信号准备层，不包括交易成本、借券与融资、容量或 walk-forward 回测。运行第一个代码单元时，会在 runs/p04_signal_layer_<uuid>/ 下新建一个独立运行目录；绘图代码单元会把 PNG 写入该目录下的 figures/a5/p04_signal_layer.png。请从头按顺序运行。


In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("p04_signal_layer_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import importlib.util
import numpy as np, pandas as pd
spec = importlib.util.spec_from_file_location("p04", ROOT / "projects/project04/preparation/signal_layer.py")
p04 = importlib.util.module_from_spec(spec); spec.loader.exec_module(p04)


## 1. 冻结规则、字段契约与构造的宇宙


In [ ]:
import numpy as np
import pandas as pd

rules, contract = p04.load_rules(), p04.load_contract()
universe = p04.build_universe()
print("frame:", universe["frame"])
print("rules sha256", rules["_sha256"][:16], "| contract sha256", contract["_sha256"][:16])
v = universe["versions"]
print("filings", len(v), "| amendments", int((v["form"] == "10-K/A").sum()), "| names", len(universe["names"]), "| months", len(universe["index"]))
assert universe["frame"]["constructed"] and not universe["frame"]["has_real_company_data"]
pd.DataFrame(contract["signals"]).T[["style", "formula", "direction"]]

## 2. 七个原始字段、覆盖率与同日预处理


In [ ]:
raw = p04.compute_signals(universe)
cov = p04.coverage(raw, universe)
z = p04.preprocess(raw, universe, rules)
styles, composite, count = p04.style_scores(z, rules)
assert (composite.notna() == (count >= 2)).to_numpy().all()                          # at least two styles, never a zero-filled composite
checks = pd.DataFrame({k: {"mean |row mean|": v.mean(axis=1).abs().mean(), "mean row sd": v.std(axis=1, ddof=1).mean()} for k, v in z.items()}).T
print(cov.round(3).T.to_string())
checks.round(6)

## 3. 反例：将 period_end 当作 available_at（前视）


In [ ]:
pe = p04.counterexample_period_end(universe)
table = pd.DataFrame({k: {"share of cells that differ": v["share_of_cells_that_differ"], "leak corr, signal months Feb-Mar": v["leak_channel_corr_feb_mar"]["mean"],
                          "t": v["leak_channel_corr_feb_mar"]["t"], "leak corr, other months": v["leak_channel_corr_other_months"]["mean"]} for k, v in pe.items()}).T
assert table.loc["book_to_market", "t"] > 3 and 0.15 < table.loc["gp_a", "share of cells that differ"] < 0.25
table.round(3)

## 4. 反例：同一版本的重述处理规则


In [ ]:
rs = p04.counterexample_restatement(universe)
print({k: (round(x, 4) if isinstance(x, float) else x) for k, x in rs.items() if not isinstance(x, dict)})
print(rs["month_before_amendment"], rs["month_of_amendment"])
assert rs["month_before_amendment"]["pit_equity"] == rs["original_equity"] and rs["month_of_amendment"]["pit_equity"] == rs["amended_equity"]

## 5. 反例：未来数据扰动（截止月及之前的信号不变）


In [ ]:
fp = p04.counterexample_future_perturbation(universe, rules)
print("cutoff", fp["cutoff"], "| months compared", fp["months_compared"], "| max |difference| up to the cutoff:", max(fp["max_abs_difference_up_to_cutoff"].values()),
      "| composite after the cutoff moves by up to", round(fp["max_abs_difference_of_composite_after_cutoff"], 3))
assert max(fp["max_abs_difference_up_to_cutoff"].values()) == 0.0 and fp["max_abs_difference_of_composite_after_cutoff"] > 0.1

## 6. 反例：幸存者宇宙、非正分母与全样本标准化


In [ ]:
sv = p04.counterexample_survivorship(universe, rules)
dn = p04.counterexample_denominators(universe)
fs = p04.counterexample_full_sample_standardization(universe, rules)
print("survivors only removes", sv["names_removed_at_date"], "names at", sv["date"], "|", sv["at_date"])
print({k: v for k, v in dn.items()})
print("full-sample vs expanding z-score: opposite sign in", round(100 * fs["share_with_opposite_sign"], 1), "% of the cells")
assert dn["safe_bm_negative_cells"] == 0 < dn["naive_negative_bm_cells"] and 0.05 < fs["share_with_opposite_sign"] < 0.25

## 7. 合成规则与暴露（不含成本）；构造零效应世界中的 IC


In [ ]:
d = p04.diagnostics(universe, rules)
print(pd.DataFrame(d["ic"]).T.round(4).to_string())
pd.DataFrame({"plain quintile": d["exposures_quintile_plain"], "industry-neutral projection": d["exposures_quintile_industry_neutral"]}).round(4)

In [ ]:
from IPython.display import Image, display
figure_path = RUN_DIR / p04.FIGURE
p04.make_figure(universe, {"diagnostics":d, "counterexample_period_end":pe}, figure_path)
display(Image(filename=str(figure_path)))


## 8. 自己试一试

1. 将 rules["preprocessing"]["winsorize"]["lower"] 改为 0.05，再运行第 2 节和第 7 节。写下这项选择应由谁在什么时候确定，以及是否应记入账本。
2. 在第 5 节将截止月改为 "2016-06"，确认截止月之前的信号仍然不变。
3. 在第 3 节说明为什么 earnings_yield 和 roe_avg 的泄漏相关性接近 0，而 book_to_market 和 gp_a 的泄漏相关性不是（提示：在构造数据中，财报新闻会进入资产）。
